<a href="https://colab.research.google.com/github/si-janenunes/CURSO-IA-CETAM-2026/blob/main/ModuloII_Chatboot_simples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

"""
===========================================================
ASSISTENTE DE SUPORTE TÉCNICO - VERSÃO 1
===========================================================

Objetivo didático:
    Demonstrar como construir um chatbot simples baseado
    em regras, sem Machine Learning.

Conceitos de PLN trabalhados:
    - Normalização de texto
    - Tokenização
    - Expressões Regulares (Regex)
    - Identificação de intenções
    - Dicionários de respostas
    - Regras de decisão
    - Fallback
    - Registro de conversas (logs)

Importante:
    Nesta versão NÃO utilizamos Machine Learning.

    O chatbot não "aprende" sozinho.

    O comportamento do sistema é definido por regras
    criadas pelo programador.

Versão: 1.0
"""

In [2]:
# ==========================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ==========================================================

# 're' significa Regular Expression (Expressão Regular).
# Será utilizada para procurar padrões dentro das mensagens.
import re

# 'datetime' será utilizada para registrar data e hora
# das mensagens no arquivo de log.
from datetime import datetime


In [3]:

# ==========================================================
# 2. CONFIGURAÇÕES
# ==========================================================

# Nome do arquivo onde serão armazenadas as conversas.
ARQUIVO_LOG = "conversas.log"


In [4]:
# ==========================================================
# 3. BASE DE CONHECIMENTO DO CHATBOT
# ==========================================================

"""
Neste dicionário armazenamos as respostas associadas
às diferentes intenções.

Uma intenção representa aquilo que o usuário deseja.

Por exemplo:

    "Quero saber o horário"

possui a intenção:

    horario_atendimento
"""

RESPOSTAS = {

    "saudacao": (
        "Olá! Sou o assistente de suporte técnico. "
        "Como posso ajudar?"
    ),

    "horario_atendimento": (
        "Nosso suporte funciona de segunda a sexta-feira, "
        "das 08:00 às 17:00."
    ),

    "problema_internet": (
        "Vamos verificar sua conexão. "
        "Primeiro, confirme se o cabo de rede ou o Wi-Fi "
        "está conectado corretamente."
    ),

    "computador_nao_liga": (
        "Verifique se o equipamento está conectado à tomada "
        "e se o cabo de energia está corretamente conectado."
    ),

    "computador_lento": (
        "Um computador lento pode ter várias causas. "
        "Podemos começar verificando a quantidade de memória "
        "e os programas que estão sendo executados."
    ),

    "senha": (
        "Para problemas de senha, verifique primeiro se "
        "Caps Lock está ativado. Se o problema continuar, "
        "será necessário solicitar a redefinição da senha."
    ),

    "impressora": (
        "Verifique se a impressora está ligada, conectada "
        "ao computador e possui papel. Depois podemos "
        "verificar a fila de impressão."
    ),

    "despedida": (
        "Foi um prazer ajudar. Até mais!"
    ),

}


In [5]:
# ==========================================================
# 4. PALAVRAS-CHAVE ASSOCIADAS ÀS INTENÇÕES
# ==========================================================

"""
Aqui está uma das partes mais importantes da Versão 1.

O chatbot possui uma lista de palavras que ajudam a
identificar a intenção da mensagem.

Por exemplo:

    "internet", "wifi", "wi-fi", "conexão"

estão associadas à intenção:

    problema_internet

Observe que isso NÃO é aprendizado de máquina.

O programador está explicitamente informando ao programa
quais palavras devem ser associadas a cada intenção.
"""

PALAVRAS_CHAVE = {

    "saudacao": [
        "oi",
        "ola",
        "olá",
        "bom dia",
        "boa tarde",
        "boa noite",
        "hey"
    ],

    "horario_atendimento": [
        "horario",
        "horário",
        "atendimento",
        "hora",
        "funcionamento"
    ],

    "problema_internet": [
        "internet",
        "wifi",
        "wi-fi",
        "conexao",
        "conexão",
        "rede"
    ],

    "computador_nao_liga": [
        "não liga",
        "nao liga",
        "não ligar",
        "nao ligar",
        "ligar computador"
    ],

    "computador_lento": [
        "lento",
        "lentidão",
        "lentidao",
        "travando",
        "travado",
        "demora"
    ],

    "senha": [
        "senha",
        "password",
        "acesso",
        "login"
    ],

    "impressora": [
        "impressora",
        "imprimir",
        "impressão",
        "impressao",
        "papel"
    ],

    "despedida": [
        "tchau",
        "até mais",
        "ate mais",
        "até logo",
        "ate logo",
        "falou"
    ]
}


In [6]:
# ==========================================================
# 5. NORMALIZAÇÃO DO TEXTO
# ==========================================================

def normalizar_texto(texto):
    """
    Normaliza uma mensagem recebida pelo chatbot.

    O objetivo é reduzir algumas diferenças entre as formas
    como as pessoas escrevem.

    Exemplos:

        "OI!!!"       -> "oi"
        "  Internet " -> "internet"
        "Ola..."      -> "ola"

    Esta etapa representa uma parte importante do
    pré-processamento de linguagem natural.
    """

    # Converte todas as letras para minúsculas.
    texto = texto.lower()

    # Remove espaços no início e no final.
    texto = texto.strip()

    # Remove pontuações.

    # O padrão:
    # [^\w\s]
    #
    # significa:
    # "qualquer caractere que não seja letra, número,
    #  underscore ou espaço".
    texto = re.sub(r"[^\w\s]", "", texto)

    # Substitui vários espaços consecutivos por apenas um.
    texto = re.sub(r"\s+", " ", texto)

    return texto


In [7]:
# ==========================================================
# 6. TOKENIZAÇÃO
# ==========================================================

def tokenizar(texto):
    """
    Divide o texto em palavras (tokens).

    Exemplo:

        "meu computador está lento"

    torna-se:

        ["meu", "computador", "está", "lento"]

    Nesta primeira versão utilizamos uma tokenização simples,
    baseada nos espaços.
    """

    return texto.split()

In [8]:
# ==========================================================
# 7. DETECÇÃO DE PADRÕES COM REGEX
# ==========================================================

def detectar_numero_chamado(texto):
    """
    Exemplo didático de utilização de Expressão Regular.

    Procura um número de chamado dentro da mensagem.

    Exemplos reconhecidos:

        "chamado 12345"
        "protocolo 9876"

    Retorna o número encontrado ou None.
    """

    padrao = r"(?:chamado|protocolo)\s*(\d+)"

    resultado = re.search(padrao, texto, re.IGNORECASE)

    if resultado:
        return resultado.group(1)

    return None

In [9]:
# ==========================================================
# 8. IDENTIFICAÇÃO DA INTENÇÃO
# ==========================================================

def identificar_intencao(tokens, texto_original):
    """
    Tenta descobrir a intenção da mensagem.

    Estratégia utilizada:

        1. Verificar expressões específicas.
        2. Verificar palavras-chave.
        3. Caso nada seja encontrado, retornar None.

    Importante:
        Esta função representa uma "classificação"
        baseada em regras.

        Ela ainda NÃO utiliza Machine Learning.
    """

    # ------------------------------------------------------
    # Regra especial para problemas em que várias palavras
    # precisam aparecer juntas.
    # ------------------------------------------------------

    texto = " ".join(tokens)

    # Problema relacionado ao computador não ligar.
    if "não liga" in texto or "nao liga" in texto:
        return "computador_nao_liga"

    # ------------------------------------------------------
    # Verificação das palavras-chave.
    # ------------------------------------------------------

    for intencao, palavras in PALAVRAS_CHAVE.items():

        for palavra in palavras:

            # Se a palavra-chave possuir mais de uma palavra,
            # verificamos a expressão inteira.
            if " " in palavra:

                if palavra in texto:
                    return intencao

            # Caso seja uma palavra isolada, verificamos
            # se ela aparece entre os tokens.
            else:

                if palavra in tokens:
                    return intencao

    # Nenhuma intenção foi identificada.
    return None


In [10]:
# ==========================================================
# 9. FALLBACK
# ==========================================================

def resposta_fallback():
    """
    Executada quando o chatbot não consegue identificar
    a intenção do usuário.

    O fallback é muito importante em sistemas conversacionais.

    Um sistema real precisa saber lidar com situações
    que não foram previstas.
    """

    return (
        "Não consegui identificar o problema. "
        "Você pode tentar explicar de outra forma? "
        "Por exemplo: 'minha internet não funciona' "
        "ou 'meu computador está lento'."
    )

In [11]:
# ==========================================================
# 10. REGISTRO DE LOG
# ==========================================================

def registrar_log(mensagem_usuario, intencao, resposta):
    """
    Registra uma interação em um arquivo.

    O log permite posteriormente analisar:

        - o que o usuário perguntou;
        - qual intenção foi identificada;
        - qual resposta foi enviada;
        - quando a interação ocorreu.

    Esses dados poderão ser muito úteis nas próximas
    versões do projeto.
    """

    data_hora = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    linha = (
        f"[{data_hora}] "
        f"USUARIO: {mensagem_usuario} | "
        f"INTENCAO: {intencao} | "
        f"BOT: {resposta}\n"
    )

    with open(
        ARQUIVO_LOG,
        "a",
        encoding="utf-8"
    ) as arquivo:

        arquivo.write(linha)

In [12]:
# ==========================================================
# 11. PROCESSAMENTO DE UMA MENSAGEM
# ==========================================================

def processar_mensagem(mensagem):
    """
    Executa o pequeno pipeline de PLN da Versão 1.

    Fluxo:

        mensagem
            ↓
        normalização
            ↓
        tokenização
            ↓
        identificação da intenção
            ↓
        resposta
            ↓
        registro no log
    """

    # ------------------------------------------------------
    # ETAPA 1 - NORMALIZAÇÃO
    # ------------------------------------------------------

    texto_normalizado = normalizar_texto(mensagem)

    # ------------------------------------------------------
    # ETAPA 2 - TOKENIZAÇÃO
    # ------------------------------------------------------

    tokens = tokenizar(texto_normalizado)

    # ------------------------------------------------------
    # ETAPA 3 - IDENTIFICAÇÃO DA INTENÇÃO
    # ------------------------------------------------------

    intencao = identificar_intencao(
        tokens,
        mensagem
    )

    # ------------------------------------------------------
    # ETAPA 4 - SE A INTENÇÃO FOI ENCONTRADA
    # ------------------------------------------------------

    if intencao is not None:

        resposta = RESPOSTAS[intencao]

    # ------------------------------------------------------
    # ETAPA 5 - FALLBACK
    # ------------------------------------------------------

    else:

        intencao = "desconhecida"

        resposta = resposta_fallback()

    # ------------------------------------------------------
    # ETAPA 6 - REGISTRO
    # ------------------------------------------------------

    registrar_log(
        mensagem,
        intencao,
        resposta
    )

    return resposta

In [13]:
# ==========================================================
# 12. INTERFACE DO CHATBOT
# ==========================================================

def iniciar_chat():
    """
    Inicia uma conversa no terminal.

    O usuário pode digitar mensagens normalmente.

    Para encerrar:

        sair
        exit
        quit
    """

    print("=" * 60)
    print("       ASSISTENTE DE SUPORTE TÉCNICO")
    print("=" * 60)

    print()
    print("Olá! Sou seu assistente de suporte técnico.")
    print("Digite sua dúvida ou problema.")
    print("Para sair, digite 'sair'.")
    print()

    while True:

        # Recebe a mensagem do usuário.
        mensagem = input("Você: ")

        # Verifica se o usuário deseja encerrar.
        if mensagem.lower().strip() in [
            "sair",
            "exit",
            "quit"
        ]:

            resposta = RESPOSTAS["despedida"]

            print("Bot:", resposta)

            break

        # Processa a mensagem utilizando o pipeline.
        resposta = processar_mensagem(mensagem)

        # Exibe a resposta.
        print("Bot:", resposta)

        print()

In [ ]:
# ==========================================================
# 13. PONTO DE ENTRADA DO PROGRAMA
# ==========================================================

if __name__ == "__main__":

    iniciar_chat()


       ASSISTENTE DE SUPORTE TÉCNICO

Olá! Sou seu assistente de suporte técnico.
Digite sua dúvida ou problema.
Para sair, digite 'sair'.

Você: Meu computador esta travado
Bot: Um computador lento pode ter várias causas. Podemos começar verificando a quantidade de memória e os programas que estão sendo executados.

Você: o que faço
Bot: Não consegui identificar o problema. Você pode tentar explicar de outra forma? Por exemplo: 'minha internet não funciona' ou 'meu computador está lento'.

Você: lento
Bot: Um computador lento pode ter várias causas. Podemos começar verificando a quantidade de memória e os programas que estão sendo executados.

Você: sem internet
Bot: Vamos verificar sua conexão. Primeiro, confirme se o cabo de rede ou o Wi-Fi está conectado corretamente.

